# Arcadia morning: one dream wallpaper (Phase 1.5)

Runs `daily/morning_job.py`: run 4 dreams 12 candidates, the best is picked and "remembered" by Qwen2.5-1.5B, Qwen writes a painting prompt, SDXL paints it (canonical Corva as a style-only reference), and the text is set over it as desktop and phone wallpapers.

Launched every morning by `daily/morning.py`. Needs **GPU T4**, **Internet on**, and the private dataset **danieltgregory/arcadia-run4-model** (run 4's `best.pt` and `train.txt`) attached. Output: `/kaggle/working/morning/<date>/`.

## 1. Clone the repo

In [ ]:
import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b main https://github.com/DanielGregory/Arcadia /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing (torch is never reinstalled)

In [ ]:
import importlib.util, subprocess, sys

missing = [pkg for pkg in ("diffusers", "accelerate", "sentencepiece") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 3. Find the model dataset and check the GPU

Kaggle has mounted attached datasets at more than one path over the years, so search for the files instead of hard-coding one.

In [ ]:
import glob, torch
ckpt = glob.glob("/kaggle/input/**/run4_best.pt", recursive=True)[0]
train = glob.glob("/kaggle/input/**/train.txt", recursive=True)[0]
print(ckpt, train, sep="\n")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 4. Make the morning

`check=True`: a failure fails the whole Kaggle run, instead of finishing as "complete" with nothing made.

In [ ]:
subprocess.run([sys.executable, "-m", "daily.morning_job", "--ckpt", ckpt, "--train", train,
                "--out", "/kaggle/working/morning"], check=True)

## 5. Show it

In [ ]:
for path in glob.glob("/kaggle/working/morning/*/morning.md"):
    print(open(path).read())